In [ ]:
%pip install brian2 brian2hears librosa numpy scipy matplotlib
%pip install setuptools

In [ ]:
import logging
logging.getLogger('brian2').setLevel(logging.ERROR)

import brian2
brian2.prefs.codegen.target = 'numpy'

from brian2 import Hz, kHz
from brian2hears import Sound, erbspace, Gammatone, Filterbank
from scipy import signal
import librosa
import numpy as np
import matplotlib.pyplot as plt
from math import gcd 
import os
import glob

class EnvelopeFromGammatoneFilterbank(Filterbank):
    """Converts the output of a GammatoneFilterbank to an envelope."""

    def __init__(self, source):
        super().__init__(source)
        self.nchannels = 1

    def buffer_apply(self, input_):
        # 6. take absolute value of the input_
        abs_input = np.abs(input_)
        # 7. power-law compression (exponent 0.6)
        compressed = abs_input ** 0.6
        # 8. linearly combine by summing the subbands
        envelope = np.sum(compressed, axis=1, keepdims=True) 
        return envelope

def process_audio_file(wav_filename, mode='cnn'):
    if mode == 'linear':
        target_sr = 20
        lowcut = 1.0
        highcut = 9.0
    elif mode == 'cnn':
        target_sr = 64
        lowcut = 1.0
        highcut = 32.0
    else:
        raise ValueError("Kies 'linear' of 'cnn' als mode.")
    audio_data, sr = librosa.load(wav_filename, sr=None, mono=True)

    # 2. convert the audio file to a brian sound object
    sound = Sound(audio_data.reshape(-1, 1), samplerate=sr*Hz) #audio in 1 kolom (kanaal) door reshape vooor brian


    # 3. 28 center frequencies between 50 Hz and 5 kHz
    cf = erbspace(50*Hz, 5*kHz, 28)

    # 4. create the gammatone filterbank
    gammatone_filterbank = Gammatone(sound, cf)

    # 5. process envelope
    envelope_calcuation = EnvelopeFromGammatoneFilterbank(gammatone_filterbank)
    envelope = envelope_calcuation.process()
    envelope = envelope.flatten() 

    # 6. Bandpass filter instellen (1-32 Hz)
    sos = signal.butter(N=4, Wn=[lowcut, highcut], btype='bandpass', fs=sr, output='sos')
    envelope_filtered = signal.sosfiltfilt(sos, envelope)

    # 7. Downsample de resulterende signalen naar 64Hz
    g = gcd(int(sr), target_sr)
    envelope_downsampled = signal.resample_poly(envelope_filtered, target_sr // g, int(sr) // g)
    
    return envelope_downsampled
def process_eeg_file(npz_filename, mode='cnn'):
    
    data = np.load(npz_filename)
    eeg_data = data['eeg']  # Vorm: (116736, 64)
    fs = int(data['fs'])    # 128 Hz
    
    #haal de paden van de bijbehorende audio op
    attended_wav = str(data['stimulus_attended'])
    unattended_wav = str(data['stimulus_unattended'])

    if mode == 'linear':
        target_sr = 20
        lowcut = 1.0
        highcut = 9.0
    elif mode == 'cnn':
        target_sr = 64
        lowcut = 1.0
        highcut = 32.0
    else:
        raise ValueError("Kies 'linear' of 'cnn' als mode.")

    # fs is hier de fs van de EEG (128 Hz)
    sos = signal.butter(N=4, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')
    
    # voeg axis=0 toe, zodat hij filtert over de tijd, niet over de 64 kanalen heen
    eeg_filtered = signal.sosfiltfilt(sos, eeg_data, axis=0)

    # downsample
    g = gcd(fs, target_sr)
    # Ook hier axis=0 toevoegen
    eeg_downsampled = signal.resample_poly(eeg_filtered, target_sr // g, fs // g, axis=0)
    
    
    return eeg_downsampled, attended_wav, unattended_wav



In [ ]:
def create_lagged_eeg(eeg_data, num_lags):
   
    n_samples, n_channels = eeg_data.shape
    
  
    valid_samples = n_samples - num_lags + 1
    
    lagged_matrix = np.zeros((valid_samples, n_channels * num_lags))
    
    for lag in range(num_lags):
        start_col = lag * n_channels
        end_col = (lag + 1) * n_channels
        
        lagged_matrix[:, start_col:end_col] = eeg_data[lag : lag + valid_samples, :]
        
    return lagged_matrix



In [ ]:
def calculate_covariance_matrices(lagged_eeg, attended_envelope):

    min_length = min(lagged_eeg.shape[0], attended_envelope.shape[0])
    
    M = lagged_eeg[:min_length, :]
    S = attended_envelope[:min_length]
    
    
    R = np.dot(M.T, M) 
    
    
    r_ms = np.dot(M.T, S)
    
    return R, r_ms, min_length



In [ ]:
from scipy.stats import pearsonr
import numpy as np

def run_leave_one_out_windowed(R_matrices, rms_vectoren, trials_info, window_sec=10, fs=20):
    
    correcte_windows = 0
    totaal_windows = 0
    
    window_samples = window_sec * fs # Bijv 10 sec * 20 Hz = 200 samples
    totaal_trials = len(R_matrices)
    
    print(f"\nStart Windowed Evaluatie ({window_sec} seconden per beslissing)...")
    
    for k in range(totaal_trials):
        R_train = sum([R_matrices[i] for i in range(totaal_trials) if i != k])
        rms_train = sum([rms_vectoren[i] for i in range(totaal_trials) if i != k])
        decoder_d = np.dot(np.linalg.pinv(R_train), rms_train)
        
        test_info = trials_info[k]
        
        eeg_test, _, _ = process_eeg_file(test_info['eeg_pad'], mode='linear')
        lagged_test_eeg = create_lagged_eeg(eeg_test, num_lags=5)
        
        att_env_test = np.load(test_info['att_audio_pad'])
        unatt_env_test = np.load(test_info['unatt_audio_pad'])
        
        min_len = min(lagged_test_eeg.shape[0], att_env_test.shape[0], unatt_env_test.shape[0])
        gereconstrueerde_audio = np.dot(lagged_test_eeg[:min_len, :], decoder_d)
        
        aantal_blokjes = min_len // window_samples
        
        trial_correct = 0
        
        for w in range(aantal_blokjes):
            start_idx = w * window_samples
            eind_idx = start_idx + window_samples
            
            recon_chunk = gereconstrueerde_audio[start_idx:eind_idx]
            att_chunk = att_env_test[start_idx:eind_idx]
            unatt_chunk = unatt_env_test[start_idx:eind_idx]
            
            r_att, _ = pearsonr(recon_chunk, att_chunk)
            r_unatt, _ = pearsonr(recon_chunk, unatt_chunk)
            
            if r_att > r_unatt:
                correcte_windows += 1
                trial_correct += 1
            totaal_windows += 1
            
        print(f" Trial {k+1}: {trial_correct}/{aantal_blokjes} windows correct ({trial_correct/aantal_blokjes*100:.1f}%)")

    accuracy = (correcte_windows / totaal_windows) * 100
    return accuracy



In [ ]:
import os
import glob
import numpy as np


eeg_hoofdmap = "data/64 Channel Biosemi unprocessed data - train/"

audio_hoofdmap = "data/preprocessed/audio/linear/" 
mode = 'linear'
num_lags = 5

alle_scores = {}

subject_mappen = glob.glob(os.path.join(eeg_hoofdmap, "sub-*", "sub-*"))
subject_mappen.sort() 

print(f"Gevonden proefpersonen in totaal: {len(subject_mappen)}")

for subject_pad in subject_mappen:
    subject_naam = os.path.basename(subject_pad)
    
    
    print(f" STARTEN MET PROEFPERSOON: {subject_naam}")
    
    eeg_bestanden = glob.glob(os.path.join(subject_pad, "*.npz"))
    
    if len(eeg_bestanden) < 2:
        print(f"Niet genoeg trials voor {subject_naam}.")
        continue
        
    R_matrices = []
    rms_vectoren = []
    alle_trials_info = []
    
    for i, eeg_pad in enumerate(eeg_bestanden):
        eeg_processed, att_naam, unatt_naam = process_eeg_file(eeg_pad, mode=mode)
        
        att_npy_naam = att_naam.replace('.wav', '.npy')
        unatt_npy_naam = unatt_naam.replace('.wav', '.npy')
        

        zoek_att = glob.glob(os.path.join(audio_hoofdmap, "**", att_npy_naam), recursive=True)
        zoek_unatt = glob.glob(os.path.join(audio_hoofdmap, "**", unatt_npy_naam), recursive=True)
        
        if len(zoek_att) == 0 or len(zoek_unatt) == 0:
            print(f"  Audio {att_npy_naam} of {unatt_npy_naam} NIET gevonden. Trial {i+1} overgeslagen!")
            continue
            
        att_audio_pad = zoek_att[0] 
        unatt_audio_pad = zoek_unatt[0]
        
        att_envelope = np.load(att_audio_pad)
        lagged_eeg = create_lagged_eeg(eeg_processed, num_lags)
        R_trial, rms_trial, lengte = calculate_covariance_matrices(lagged_eeg, att_envelope)
        
        R_matrices.append(R_trial)
        rms_vectoren.append(rms_trial)
        
        alle_trials_info.append({
            'eeg_pad': eeg_pad,
            'att_audio_pad': att_audio_pad,
            'unatt_audio_pad': unatt_audio_pad 
        })

    if len(R_matrices) < 2:
        print(f"Te weinig geldige data voor {subject_naam}. LOO onmogelijk.")
        continue

    print(f"Start 10s Window Evaluatie voor {subject_naam}...")
    score = run_leave_one_out_windowed(R_matrices, rms_vectoren, alle_trials_info, window_sec=10)
    
    alle_scores[subject_naam] = score
    print(f" Eindscore voor {subject_naam}: {score:.1f}%")

if len(alle_scores) > 0:
    gemiddelde_score = np.mean(list(alle_scores.values()))
    print(f"EINDRESULTAAT OVER {len(alle_scores)} PROEFPERSONEN")
    print(f"Gemiddelde Nauwkeurigheid (10s): {gemiddelde_score:.1f}%")
